# Testing Pipeline for the `numpandas` package

In [1]:
import json
import tempfile
from pathlib import Path

import numpy as np

import numpandas as npd


def assert_raises(exc_type, func, *args, **kwargs):
    try:
        func(*args, **kwargs)
    except exc_type:
        return True
    except Exception as exc:
        raise AssertionError(
            f"Expected {exc_type.__name__}, got {type(exc).__name__}: {exc}"
        ) from exc
    raise AssertionError(f"Expected {exc_type.__name__} to be raised.")

**Index**
- Create from a list of labels
- Default integer index creation
- Access by position and label
- Duplicate label behavior (should it raise or allow?)

In [2]:
idx = npd.Index(["a", "b", "c"])
assert len(idx) == 3
assert idx.get_loc("b") == 1
assert idx.to_list() == ["a", "b", "c"]

s_default = npd.Series([10, 20, 30])
assert s_default.index.to_list() == [0, 1, 2]

idx_dup = npd.Index(["x", "x"])
assert_raises(ValueError, idx_dup.get_loc, "x")

True

**Series**
- Create from list, dict, and ndarray
- Access element by label and position
- Boolean mask filtering
- `map(func)` applies correctly element-wise
- NaN handling: `isna()`, `fillna()`, `dropna()`
- All stat methods: `mean`, `std`, `min`, `max`, `sum`, `count`, `median`, `var`
- `astype` with safe and unsafe casts (unsafe should raise)

In [3]:
s = npd.Series([1, 2, np.nan, 4], name="x")
assert s.name == "x"
assert s.to_list()[0] == 1
assert np.isnan(s.to_list()[2])
assert s.loc[0] == 1
assert s.iloc[1] == 2

mask = s.isna()
assert mask.to_list() == [False, False, True, False]

filled = s.fillna(0)
assert filled.to_list() == [1, 2, 0, 4]

dropped = s.dropna()
assert dropped.to_list() == [1, 2, 4]

mapped = s.map(lambda v: v * 2)
assert np.isnan(mapped.to_list()[2])
assert mapped.to_list()[0] == 2

assert s.count() == 3
assert np.isclose(s.sum(), 7.0)
assert np.isclose(s.mean(), 7 / 3)
assert np.isclose(s.min(), 1.0)
assert np.isclose(s.max(), 4.0)
assert np.isclose(s.median(), 2.0)

vals = np.array([1, 2, 4], dtype=float)
assert np.isclose(s.std(), np.std(vals, ddof=1))
assert np.isclose(s.var(), np.var(vals, ddof=1))

assert_raises(ValueError, s.astype, int)

True

**DataFrame**
- Create from dict of lists/arrays
- `df[col]` returns a Series
- `df[[col1, col2]]` returns a DataFrame
- `df.loc[label, col]` and `df.iloc[int, int]` correct values
- Boolean mask filtering `df[bool_series]`
- `isna()` returns correct boolean DataFrame
- `fillna(scalar)` and `fillna({col: value})` per-column
- `dropna(axis=0, how='any')`, `dropna(axis=0, how='all')`, `dropna(axis=1, how='any')`, `dropna(axis=1, how='all')`
- `astype({col: dtype})` safe and unsafe
- `apply(func, axis=0)` and `apply(func, axis=1)`
- `sample(n=...)`, `sample(frac=...)`, `sample(random_state=...)` reproducibility
- `head(n)` and `tail(n)`
- `rename(columns={...})` returns new object, original unchanged (CoW)
- `drop(columns=[...])` returns new object
- `reset_index()` resets to integer range
- `describe()` returns correct summary stats
- `shape`, `columns`, `dtypes` properties
- `from_numpy(array, columns)` constructor
- `to_numpy()` round-trip with `from_numpy`

In [4]:
df = npd.DataFrame(
    {"a": [1, 2, np.nan], "b": [10, 20, 30], "c": ["x", "y", "z"]},
    index=["r1", "r2", "r3"],
)

assert df.shape == (3, 3)
assert df.columns == ["a", "b", "c"]
assert df["a"].to_list()[1] == 2
subset = df[["a", "b"]]
assert subset.shape == (3, 2)

assert df.loc["r2", "b"] == 20
assert df.iloc[0, 1] == 10

mask = npd.Series([True, False, True], index=["r1", "r2", "r3"])
filtered = df[mask]
assert filtered.shape == (2, 3)
assert filtered.index.to_list() == ["r1", "r3"]

isna = df.isna()
assert isna["a"].to_list() == [False, False, True]

filled = df.fillna(0)
assert filled["a"].to_list() == [1, 2, 0]

filled_col = df.fillna({"a": 5})
assert filled_col["a"].to_list() == [1, 2, 5]

drop_any = df.dropna(axis=0, how="any")
assert drop_any.index.to_list() == ["r1", "r2"]
drop_all = df.dropna(axis=0, how="all")
assert drop_all.shape == (3, 3)
drop_col_any = df.dropna(axis=1, how="any")
assert drop_col_any.columns == ["b", "c"]
drop_col_all = df.dropna(axis=1, how="all")
assert drop_col_all.columns == ["a", "b", "c"]

casted = df.astype({"b": "float64"})
assert casted.dtypes["b"] == np.dtype("float64")
assert_raises(ValueError, df.astype, {"a": "int64"})

col_counts = df.apply(lambda s: s.count(), axis=0)
assert col_counts.to_list() == [2, 3, 3]

row_sum = df.apply(lambda r: r.iloc[0] + r.iloc[1], axis=1)
assert row_sum.to_list()[0] == 11

sample1 = df.sample(n=2, random_state=42)
sample2 = df.sample(n=2, random_state=42)
assert np.array_equal(sample1.to_numpy(), sample2.to_numpy())

assert df.head(2).shape == (2, 3)
assert df.tail(1).index.to_list() == ["r3"]

renamed = df.rename(columns={"a": "a1"})
assert renamed.columns == ["a1", "b", "c"]
assert df.columns == ["a", "b", "c"]

dropped = df.drop(columns=["c"])
assert dropped.columns == ["a", "b"]

reset = df.reset_index()
assert reset.index.to_list() == [0, 1, 2]

desc = df.describe()
assert desc.columns == ["a", "b"]
assert "mean" in desc.index.to_list()

arr = df[["a", "b"]].to_numpy()
round_trip = npd.DataFrame.from_numpy(arr, columns=["a", "b"])
assert round_trip.shape == (3, 2)

**I/O**
- `read_csv` → `to_csv` round-trip (values match)
- `read_json` → `to_json` round-trip
- `read_excel` → `to_excel` round-trip
- CSV with missing values loads as `np.nan` (not `None` or empty string)
- Reading a file with mixed types infers correctly or raises

In [5]:
with tempfile.TemporaryDirectory() as tmp_dir:
    tmp_path = Path(tmp_dir)
    df_io = npd.DataFrame({"a": [1, np.nan, 3], "b": [10, 20, 30]})

    csv_path = tmp_path / "test.csv"
    df_io.to_csv(str(csv_path), index=False)
    df_csv = npd.read_csv(str(csv_path))
    assert df_csv.columns == ["a", "b"]
    assert np.isnan(df_csv["a"].to_list()[1])

    json_path = tmp_path / "test.json"
    df_io.to_json(str(json_path))
    df_json = npd.read_json(str(json_path))
    assert np.isnan(df_json["a"].to_list()[1])

    try:
        excel_path = tmp_path / "test.xlsx"
        df_io.to_excel(str(excel_path))
        df_excel = npd.read_excel(str(excel_path))
        assert df_excel.columns == ["a", "b"]
        assert np.isnan(df_excel["a"].to_list()[1])
    except ImportError:
        print("openpyxl not installed; skipping Excel tests")

**CoW enforcement**
- Modifying the result of any operation does not affect the original DataFrame
- No method accepts or has an `inplace` parameter

In [6]:
df_cow = npd.DataFrame({"a": [1, 2], "b": [3, 4]})
renamed = df_cow.rename(columns={"a": "x"})
dropped = renamed.drop(columns=["b"])

assert df_cow.columns == ["a", "b"]
assert renamed.columns == ["x", "b"]
assert dropped.columns == ["x"]

try:
    df_cow.rename(columns={"a": "y"}, inplace=True)
    raise AssertionError("Expected TypeError for inplace parameter.")
except TypeError:
    pass

**Error handling**
- `df['nonexistent_col']` raises a clear error
- `df.astype({col: 'int'})` on a column with NaN raises with a descriptive message
- `df[mask]` where mask length doesn't match raises
- `fillna` with wrong type raises

In [7]:
df_err = npd.DataFrame({"a": [1, np.nan], "b": [2, 3]})
assert_raises(KeyError, df_err.__getitem__, "missing")
assert_raises(ValueError, df_err.astype, {"a": "int64"})

bad_mask = npd.Series([True], index=[0])
assert_raises(ValueError, df_err.__getitem__, bad_mask)

assert_raises(KeyError, df_err.fillna, {"c": 0})

True